In [1]:
# dementia_qwen_predict_fewshot_noreasoning.py
import os
import re
import sys
from pathlib import Path
import torch
import pandas as pd
import csv
from typing import List, Dict, Optional
from huggingface_hub import HfApi, snapshot_download
from transformers import AutoTokenizer, AutoModelForCausalLM

# Optional: TF-IDF retrieval for few-shot
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# ----------------- CONFIG -----------------
DATA_DIR = r"D:\Dementia-Thesis - Don't access without permission\Thesis\cookie_control_dementia"
MODEL_NAME = "Qwen/Qwen3-4B"
HF_TOKEN = os.environ.get("HF_TOKEN")
OUTPUT_CSV = r"D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass.csv"

USE_FEWSHOT = True
FEWSHOT_K = 3
EXAMPLES_CSV: Optional[str] = None  # if you have labeled examples CSV

GEN_MAX_NEW_TOKENS = 32768
GEN_TEMPERATURE = 0.3
GEN_TOP_P = 0.9
ENABLE_THINKING = True  # Enable Qwen3 thinking mode

ENHANCED_SYSTEM_PROMPT = """## DEMENTIA SCREENING: COOKIE-THEFT PICTURE DESCRIPTION TEST (CDPDT)

**TASK:** Analyze Cookie-Theft transcripts and classify as **Control** (cognitively normal) or **ProbableAD** (Alzheimer's disease).  
Provide a binary diagnosis + estimated MMSE score (0-30).

---

## DECISION FRAMEWORK: CONTROL vs PROBABLEAD

### Quick Decision Rules

**CONTROL (Cognitively Normal):**
- Clear, organized description with logical flow
- Minimal word-finding pauses (<10%)
- High semantic content (describes most key elements)
- Maintains coherent narrative structure
- Few repetitions or self-corrections
- MMSE typically 27-30

**PROBABLE AD (Alzheimer's Disease):**
- Fragmented or disorganized narrative
- Frequent pausing/hesitations (>15% of utterance)
- Missing key story elements or relationships
- Poor sequence/temporal organization
- Multiple repetitions or rambling
- Reduced detail and anomia (word-finding difficulty)
- MMSE typically 10-24

---

## KEY DISCRIMINATIVE FEATURES (RANKED BY IMPORTANCE)

### 1. **SEMANTIC COMPLETENESS & RELEVANCE** (Most Important)
Use this scoring:

**Control Pattern:**
- Identifies core objects: mother, child/boy, girl, cookie jar, stool, water, sink, window
- Describes key action: boy reaching for/taking cookies, boy on unstable stool
- Notes consequences: water overflowing, mother at sink, risk of falling
- ~12-14 relevant units mentioned

**ProbableAD Pattern:**
- Misses central action or relationships ("there's a boy... and a stool...")
- Omits critical elements (no mention of cookie jar, or mother, or consequences)
- Focuses on minor details ("the curtains are blue") while missing main events
- ~3-9 relevant units mentioned
- May confabulate details not in picture

**→ Red Flag:** If >3 major elements (jar, boy reaching, stool, water, mother) are completely missing = strong AD indicator

---

### 2. **SPEECH FLUENCY & PAUSING** (Very Important)

**Control Pattern:**
- Smooth, continuous speech with minimal hesitation
- Occasional natural pauses but quick recovery
- <10% of total time spent pausing
- Few "um/uh" or "like" fillers

**ProbableAD Pattern:**
- Frequent long pauses (>1-2 seconds) before key words
- Repetitive fillers: "um... um... the... the..."
- Multiple false starts: "He... she... I mean the boy..."
- Visible struggle to retrieve common words ("cookie... that... you know, the sweet thing")
- 20-40% of speech is pausing/hesitation

**→ Red Flag:** If describing simple objects requires multiple pauses and restarts = word-finding difficulty (anomia)

---

### 3. **NARRATIVE STRUCTURE & COHERENCE** (Very Important)

**Control Pattern:**
- Information presented in logical sequence (first describes mother, then child, then action)
- Uses temporal markers: "while", "and then", "as", "because"
- Connections between events are clear
- Topic stays focused on picture

**ProbableAD Pattern:**
- Jumbled order (describes actions out of sequence)
- Missing temporal/causal connectors
- Topic drift ("reminds me of my own kitchen", "I had a friend...")
- Contradictions or confused pronouns ("she... he... it is...")
- Rambling: idea doesn't flow logically

**→ Red Flag:** If narrative is hard to follow or jumps around = executive dysfunction (hallmark of AD)

---

### 4. **REPETITION & PERSEVERATION** (Important)

**Control Pattern:**
- Mentions items once or twice
- Self-corrections are rare and natural ("the boy, or actually the two children...")
- <2 repetitions of the same word/phrase

**ProbableAD Pattern:**
- Repeats the same word excessively: "the cookie... cookie... cookies... cookie jar..."
- Repeats entire phrases: "water running, water running, the water is running..."
- Perseverates on minor details: "the curtains... the curtains are open... the curtains are blowing..."
- 5-10+ repetitions

**→ Red Flag:** Excessive repetition of single concept = semantic retrieval deficit or working memory impairment

---

### 5. **DETAIL & ACCURACY** (Important)

**Control Pattern:**
- Accurate object labels ("stool", "faucet", "dishcloth")
- Correct relationships: "boy on stool", "mother drying dishes"
- Accurate scene interpretation: "boy is reaching/stealing cookies"

**ProbableAD Pattern:**
- Vague labels ("the thing", "stuff", "it")
- Mislabeling or generic terms instead of specific objects
- Incorrect relationships: "the boy is sitting" (when he's reaching), "the girl is eating"
- Semantic errors: confuses who is doing what

**→ Red Flag:** Inability to name common objects or misunderstanding simple relationships = semantic decline

---

## SCORING CHECKLIST FOR BINARY DECISION

Count how many of these abnormal indicators are present:

| Feature | Abnormal Threshold | Status |
|---------|-------------------|--------|
| Semantic completeness | <9 relevant units | ☐ |
| Pausing frequency | >15% of utterance | ☐ |
| Narrative incoherence | Hard to follow flow | ☐ |
| Excessive repetition | >4 repetitions of same element | ☐ |
| Anomia/vague terms | >5 instances ("thing", "stuff") | ☐ |
| Missing temporal markers | Uses <2 temporal words | ☐ |
| Self-corrections | >3 false starts | ☐ |
| Topic drift | >1 off-topic tangent | ☐ |

**DECISION RULE:**
- **0-1 abnormal indicators** → **CONTROL** (MMSE: 27-30)
- **2-3 abnormal indicators** → **CONTROL** (MMSE: 25-27) [borderline, slight decline]
- **4-5 abnormal indicators** → **PROBABLE AD** (MMSE: 18-24) [mild-moderate AD]
- **6+ abnormal indicators** → **PROBABLE AD** (MMSE: 10-17) [moderate-severe AD]

---

## MMSE ESTIMATION GUIDE

**MMSE 27-30 (Control):**
- Clear, complete description; minimal pauses; coherent; accurate
- All major elements present; good temporal organization

**MMSE 24-26 (Borderline/Mild):**
- Mostly coherent; occasional pauses; minor omissions; 1-2 vague terms

**MMSE 18-23 (Mild-Moderate AD):**
- Visible word-finding difficulty; some disorganization; missing 2-3 key elements
- Multiple pauses; repetitions present

**MMSE 10-17 (Moderate-Severe AD):**
- Severe pausing; fragmented; multiple omissions; heavy reliance on vague terms
- Poor coherence; excessive repetition; confusion about relationships

**MMSE <10 (Severe AD):**
- Minimal coherent description; severe anomia; unable to maintain topic
- Perseveration dominant; confabulation or hallucinations present

---

## WHAT TO IGNORE (Not Relevant for Control vs AD)

- Accent, language fluency (non-native speakers can still be cognitively normal)
- Background noise or transcription artifacts
- Other dementia types (PPA, FTD, vascular) - focus only on AD vs Control
- Personality or emotional tone
- Long pauses due to thinking carefully (vs. pauses due to word-finding struggle)

---

## ANALYSIS PROCESS

1. **Read the full transcript** to understand overall coherence
2. **Count relevant units** (key elements described: mother, boy, girl, jar, stool, water, sink, action, consequence)
3. **Assess pausing** (calculate approximate % of hesitation)
4. **Check narrative flow** (is it logical? are temporal markers present?)
5. **Count repetitions** (any words/phrases mentioned >4 times?)
6. **Evaluate word choice** (specific terms vs. vague "thing"/"stuff"?)
7. **Identify anomia indicators** (word-finding struggles, restarts, circumlocution)
8. **Apply decision rule** (count abnormal indicators; classify based on threshold)
9. **Estimate MMSE** (map severity to score range)

---

## SUMMARY OF AD vs CONTROL DIFFERENTIATION

| Dimension | CONTROL | PROBABLE AD |
|-----------|---------|------------|
| **Overall Impression** | Organized, fluent, coherent | Fragmented, halting, disorganized |
| **Key Indicator** | Describes main action clearly | Struggles to convey main action |
| **Pausing** | Minimal, natural | Frequent, labored |
| **Vocabulary** | Specific, accurate | Vague, generic, anomic |
| **Coherence** | Logical flow, temporal markers | Jumbled, missing connectors |
| **Detail Level** | 12-14 relevant units | 3-9 relevant units |
| **Repetition** | <2 per concept | 5+ per concept |
| **Typical MMSE** | 27-30 | 10-24 |

---

**Apply this framework rigorously when analyzing each transcript.**  
**Be conservative: if features are ambiguous, err toward CONTROL.**  
**Use MMSE as a continuous estimate (not binary) to reflect severity.**"""

# ----------------- BUILTIN FEW-SHOT EXAMPLES (no reasoning) -----------------
BUILTIN_EXAMPLES: List[Dict] = [
    {
        "filename": "002-0.txt",
        "transcript": """the scene is <in the> SFALSESTART in the kitchen
the mother is wiping dishes and the water is running on the floor
<a child is tryin to get> LFALSESTART a boy is tryin to get cookies outta a jar and he is about to tip over on a stool
PAUSE the little girl is reacting to his falling
PAUSE it seems to be summer out
the window is open
the curtains are blowing
it must be a gentle breeze
there is grass outside in the garden
PAUSE mother is finished certain of the SFALSESTART the dishes
kitchen is very tidy
the mother seems to have nothing in the house to eat except cookies in the cookie jar
PAUSE the children look to be almost about the same size
perhaps they are twins
they are dressed for summer warm weather
PAUSE you want more EXC
RETRACE the mother is in a short sleeve dress
RETRACE I will hafta say it is warm EXC""",
        "label": "Control",
        "mmse": 30
    },
    {
        "filename": "001-0.txt",
        "transcript": """mhm EXC
RETRACE alright EXC
there is PAUSE a young boy that is getting a cookie jar
and it LFALSESTART he is PAUSE in bad shape because PAUSE the thing is fallin over
and in the picture the mother is washin dishes and does not see it
and so <is the> LFALSESTART the water is overflowing in the sink
and the dishes might <get falled over if you do not> LFALSESTART fell LFALSESTART fall over there SFALSESTART there if you do not get it
and it LFALSESTART there LFALSESTART it is a picture of a kitchen window
and the curtains are very PAUSE distinct
but the water is STUTTER still flowing""",
        "label": "ProbableAD",
        "mmse": 18
    },
    {
        "filename": "034-1.txt",
        "transcript": """boy PAUSE taking cookies out of a cookie jar
the stool is falling
the little girl is reaching
water is running out o the faucet
the water is overflowing the sink
woman is <washin dishes> LFALSESTART LoPause drying dishes
there is nothing to indicate XXX and I do not see any more action EXC
""",
        "label": "Control",
        "mmse": 28
    },
    {        
        "filename": "004-0.txt",
        "transcript": """now honey I STUTTER STUTTER STUTTER STUTTER STUTTER had it was in the kitchen and I was the oldest ten EXC
and if we made a mess like that you would get a kick in the ass EXC
RETRACE well SPECIAL we have PAUSE spillin of the water
and a kid with his cookie jar
and a stool is turned over
and a mother is runnin the water on the floor
and what else do you want from that EXC
it STUTTER looks like somebody is layin out in the grass does not it
RETRACE and a kid in the cookie jar
and a tilted STUTTER stool
what more do you want EXC
RETRACE the SFALSESTART STUTTER the water rollin on the floor""",
        "label": "ProbableAD",
        "mmse": 16
    },
    {
        "filename": "684-0.txt",
        "transcript": """PAUSE the boy is reaching into the cookie jar
he is falling off the stool
the little girl is reaching for a cookie
mother is drying the dishes
the sink is running over
mother is getting her feet wet
they all have shoes on
there is <a cup> LFALSESTART two cups and a saucer on the sink
the window has draw LFALSESTART withdrawn drapes
you look out on the driveway
there is kitchen cabinets
oh what is happening EXC
mother is looking out the window
the girl is touching her lips
the boy is standing on his right foot
his left foot is sort of up in the air
mother is right foot is flat on the floor and <her left> LFALSESTART she is on her left toe
PAUSE she is holding the dish cloth in her right hand and the plate she is drying in her left
I think I have run out of EXC
RETRACE yeah EXC
""",
        "label": "Control",
        "mmse": 30
    },
    {
        "filename": "006-0.txt",
        "transcript": """oh SPECIAL you want me to tell you EXC
the mother and her two children
and the children are getting in the cookie jar
and she is doing the dishes and spilling the water
and she had the spigot on
and she did not know it perhaps
pardon me EXC
and they are looking out into the garden from the kitchen window
it is open
and the PAUSE cookies must be pretty good they are eating
<the tair > LFALSESTART PAUSE the chair is STUTTER PAUSE tilting and he is gonna fall off
and PAUSE <the lady> LFALSESTART the mother is splashing her shoes and stockings all up overflowing the water
and there is PAUSE PAUSE a window and curtains on the window
and I can see some trees outside there
and SFALSESTART and there is dishes that STUTTER had been washed
and she is dryin them
and there is some shrub out there and TRAILOFF
""",
        "label": "ProbableAD",
        "mmse": 25
    }
]

# ----------------- HF SANITY CHECK -----------------
def hf_sanity_check(model_name: str, token: Optional[str] = None) -> bool:
    api = HfApi()
    try:
        info = api.model_info(model_name, token=token)
        print(f"HF CHECK OK: {info.modelId} (private={info.private})")
        return True
    except Exception as e:
        print(f"HF CHECK ERROR: {type(e).__name__}: {e}", file=sys.stderr)
        return False

# ----------------- TOKENIZER / MODEL LOADERS -----------------
def load_qwen_tokenizer(model_name: str, hf_token: Optional[str] = None):
    kwargs = {"trust_remote_code": True}
    if hf_token:
        kwargs["token"] = hf_token
    try:
        tok = AutoTokenizer.from_pretrained(model_name, **kwargs)
    except Exception as e_remote:
        print(f"[Tokenizer] remote load failed: {type(e_remote).__name__}: {e_remote}", file=sys.stderr)
        local_dir = snapshot_download(repo_id=model_name, token=hf_token)
        tok = AutoTokenizer.from_pretrained(local_dir, trust_remote_code=True, local_files_only=True)
    return tok

def load_qwen_model(model_name: str, hf_token: Optional[str] = None):
    kwargs = {
        "trust_remote_code": True,
        "torch_dtype": "auto",
        "device_map": "auto"
    }
    if hf_token:
        kwargs["token"] = hf_token
    try:
        return AutoModelForCausalLM.from_pretrained(model_name, **kwargs)
    except Exception as e_remote:
        print(f"[Model] remote load failed: {type(e_remote).__name__}: {e_remote}", file=sys.stderr)
        local_dir = snapshot_download(repo_id=model_name, token=hf_token)
        return AutoModelForCausalLM.from_pretrained(
            local_dir, 
            trust_remote_code=True, 
            local_files_only=True, 
            torch_dtype="auto", 
            device_map="auto"
        )

# ----------------- PROMPT / PARSING -----------------
def extract_patient_info(filename: str):
    match = re.match(r'(\d{3})', filename)
    return match.group(1) if match else None

def create_prediction_prompt(filename: str, transcript: str) -> str:
    return f"""You are an expert neuropsychologist specializing in dementia assessment. Analyze the following Cookie Theft picture description transcript.

**File:** {filename}

**Transcript:**
{transcript}

**Task:**
Based on this transcript, predict:

1. **Diagnosis Label**: Choose ONE from:
   - ProbableAD
   - Control

2. **MMSE Score**: Predict a score between 0-30.

3. **Reasoning**: Provide a brief explanation for your prediction, including linguistic markers observed.

**Response Format (only this, no extra text):**
Label: [ProbableAD/Control]
MMSE: [0-30]
Confidence: [Low/Medium/High]
Reasoning: [Your explanation here - mention specific linguistic features like word-finding difficulties, pauses, false starts, coherence, vocabulary richness, etc.]"""

def parse_llm_response(response: str) -> Dict:
    label_match = re.search(r'Label:\s*(\w+)', response)
    mmse_match = re.search(r'MMSE:\s*(\d+)', response)
    conf_match = re.search(r'Confidence:\s*(\w+)', response)
    # Extract reasoning - capture everything after "Reasoning:" until end or next field
    reasoning_match = re.search(r'Reasoning:\s*(.+?)(?:\n(?:Label|MMSE|Confidence):|$)', response, re.DOTALL)
    return {
        'label': label_match.group(1) if label_match else 'Unknown',
        'mmse': int(mmse_match.group(1)) if mmse_match else -1,
        'confidence': conf_match.group(1) if conf_match else 'Unknown',
        'reasoning': reasoning_match.group(1).strip() if reasoning_match else 'No reasoning provided',
        'raw': response
    }

# ----------------- FEWSHOT LOGIC -----------------
def load_examples_from_csv(csv_path: str) -> List[Dict]:
    examples = []
    with open(csv_path, newline='', encoding='utf-8') as fh:
        reader = csv.DictReader(fh)
        for row in reader:
            transcript = (row.get("transcript") or row.get("text") or "").strip()
            label = (row.get("label") or "").strip()
            mmse = row.get("mmse") or ""
            examples.append({
                "filename": row.get("filename") or "example",
                "transcript": transcript,
                "label": label,
                "mmse": int(mmse) if str(mmse).isdigit() else None
            })
    return examples

def pick_k_most_similar_examples(query: str, examples: List[Dict], k: int) -> List[Dict]:
    if not examples:
        return []
    if SKLEARN_AVAILABLE and len(examples) > 1:
        docs = [ex["transcript"] for ex in examples]
        vect = TfidfVectorizer(max_features=4096, stop_words='english').fit(docs + [query])
        doc_mat = vect.transform(docs)
        q_vec = vect.transform([query])
        sims = cosine_similarity(q_vec, doc_mat)[0]
        idxs = sims.argsort()[::-1][:k]
        return [examples[i] for i in idxs]
    return examples[:k]

def build_fewshot_block(examples: List[Dict]) -> str:
    if not examples:
        return ""
    parts = []
    for ex in examples:
        parts.append(
            f"--- Example: {ex.get('filename','example')} ---\n"
            f"Transcript:\n{ex.get('transcript','')}\n\n"
            f"Label: {ex.get('label','')}\nMMSE: {ex.get('mmse','')}\n"
        )
    return "\n".join(parts)

# ----------------- CHAT RENDER / GENERATION (Qwen3 style) -----------------
def generate_from_model(messages: List[Dict], tokenizer, model, enable_thinking: bool = ENABLE_THINKING) -> Dict:
    """Generate response using Qwen3 chat template with thinking mode support."""
    
    # Apply chat template with thinking mode
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )
    
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    # Generate
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=GEN_MAX_NEW_TOKENS,
        temperature=GEN_TEMPERATURE,
        top_p=GEN_TOP_P,
        do_sample=True
    )
    
    # Extract only the new tokens
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()
    
    # Parse thinking content (token 151668 is </think>)
    thinking_content = ""
    content = ""
    
    if enable_thinking:
        try:
            # Find </think> token (151668)
            index = len(output_ids) - output_ids[::-1].index(151668)
            thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
            content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")
        except ValueError:
            # No thinking token found
            content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
    else:
        content = tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
    
    return {
        "thinking": thinking_content,
        "content": content,
        "full_response": tokenizer.decode(output_ids, skip_special_tokens=True).strip("\n")
    }

# ----------------- PREDICTION FUNCTIONS -----------------
def predict_visit_fewshot(filename: str, transcript: str, tokenizer, model,
                          use_fewshot: bool = USE_FEWSHOT,
                          k: int = FEWSHOT_K,
                          examples_csv: Optional[str] = EXAMPLES_CSV,
                          builtin_examples: Optional[List[Dict]] = BUILTIN_EXAMPLES) -> Dict:
    examples = []
    if use_fewshot:
        if examples_csv and Path(examples_csv).exists():
            examples = load_examples_from_csv(examples_csv)
        else:
            examples = builtin_examples
        chosen = pick_k_most_similar_examples(transcript, examples, k)
        fewshot_block = build_fewshot_block(chosen)
    else:
        fewshot_block = ""

    main_prompt = create_prediction_prompt(filename, transcript)
    if fewshot_block:
        combined = (
            "Here are some labeled examples of Cookie Theft descriptions.\n\n"
            f"{fewshot_block}\n\n"
            "Now analyze the new transcript and predict in the same format.\n\n"
            f"{main_prompt}"
        )
    else:
        combined = main_prompt

    messages = [
        {"role": "system", "content": ENHANCED_SYSTEM_PROMPT},
        {"role": "user", "content": combined}
    ]
    
    return generate_from_model(messages, tokenizer, model)

# ----------------- MAIN -----------------
def main():
    print("Checking model on HuggingFace...")
    hf_sanity_check(MODEL_NAME, HF_TOKEN)

    print("Loading tokenizer and model...")
    tokenizer = load_qwen_tokenizer(MODEL_NAME, hf_token=HF_TOKEN)
    model = load_qwen_model(MODEL_NAME, hf_token=HF_TOKEN)
    print("Model loaded.")

    files = sorted(f for f in os.listdir(DATA_DIR) if f.endswith(".txt"))
    print(f"Found {len(files)} transcript files")

    results = []
    for i, filename in enumerate(files, 1):
        pid = extract_patient_info(filename)
        path = Path(DATA_DIR) / filename
        print(f"\n[{i}/{len(files)}] Processing {filename} (Patient ID: {pid})")
        try:
            with open(path, "r", encoding="utf-8") as f:
                transcript = f.read().strip()
            if not transcript:
                raise ValueError("Empty transcript")
            
            response = predict_visit_fewshot(filename, transcript, tokenizer, model)
            
            # Parse the content (not thinking)
            prediction = parse_llm_response(response["content"])
            
            print(f"Label: {prediction['label']} | MMSE: {prediction['mmse']} | Conf: {prediction['confidence']}")
            print(f"Reasoning: {prediction['reasoning']}")
            if response["thinking"]:
                print(f"Thinking: {response['thinking'][:200]}...")  # Show first 200 chars of thinking
            
            # Remove .txt from filename for 'id'
            file_id = re.sub(r'\.txt$', '', filename)
            
            results.append({
                "id": file_id,
                "confidence": prediction["confidence"],
                "mmse": prediction["mmse"],
                "dx": prediction["label"],
                "reasoning": prediction["reasoning"]
            })
        except Exception as e:
            print(f"ERROR: {e}", file=sys.stderr)
            file_id = re.sub(r'\.txt$', '', filename)
            results.append({
                "id": file_id,
                "confidence": "N/A",
                "mmse": -1,
                "dx": "Error",
                "reasoning": str(e)
            })

    # Save id, confidence, mmse, dx, and reasoning
    df = pd.DataFrame(results, columns=["id", "confidence", "mmse", "dx", "reasoning"])
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"\n✅ Results saved to: {OUTPUT_CSV}")

    print("\nSample predictions:")
    print(df.head())

if __name__ == "__main__":
    main()

c:\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Checking model on HuggingFace...
HF CHECK OK: Qwen/Qwen3-4B (private=False)
Loading tokenizer and model...


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.27s/it]


Model loaded.
Found 552 transcript files

[1/552] Processing 001-0.txt (Patient ID: 001)
Label: ProbableAD | MMSE: 18 | Conf: High
Reasoning: Frequent pauses ("&-uh", "&-um"), fragmented narrative with missing temporal markers, repetition of "fell" and "fall over," vague terms ("the thing"), and word-finding difficulties ("fallin(g) over"). Key elements like the boy on an unstable stool or consequences of the water overflow are underdescribed, indicating semantic incomplete and disorganized speech typical of ProbableAD.
Thinking: <think>
Okay, let's tackle this. The user provided a transcript from the Cookie-Theft Picture Description Test (CDPDT) and wants me to classify it as Control or ProbableAD, along with an MMSE score. 
...

[2/552] Processing 001-2.txt (Patient ID: 001)
Label: ProbableAD | MMSE: 18 | Conf: High
Reasoning: Frequent pauses ("&-uh", "uh"), word-finding difficulties ("I can't think of the +..."), fragmented narrative, and omissions of key elements (e.g., "mother is 

KeyboardInterrupt: 

In [ ]:
# eval_fewshot_vs_actual.py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

# ---------- Paths (as you specified) ----------
actual_path = r"D:\Dementia-Thesis - Don't access without permission\Thesis\2_classes.csv"
pred_path   = r"D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass.csv"

# ---------- Load CSVs ----------
print("Loading CSVs...")
df_true = pd.read_csv(actual_path)
df_pred = pd.read_csv(pred_path)
print("Loaded actual:", actual_path)
print("Loaded predictions:", pred_path)

# ---------- Column mapping ----------
# Actual CSV columns: id, mmse, dx (or similar)
# Prediction CSV columns: id, confidence, mmse, dx, reasoning

# For actual data - use mmse and dx columns
mmse_true_col = "mmse"
label_true_col = "dx"

# For prediction data - use mmse and dx columns
mmse_pred_col = "mmse"
label_pred_col = "dx"

print(f"Using (actual) mmse column: '{mmse_true_col}'  label column: '{label_true_col}'")
print(f"Using (pred)   mmse column: '{mmse_pred_col}'  label column: '{label_pred_col}'")

# ---------- Rename for consistency ----------
df_true = df_true.rename(columns={mmse_true_col: "actual_mmse", label_true_col: "actual_label"})
df_pred = df_pred.rename(columns={mmse_pred_col: "pred_mmse", label_pred_col: "pred_label"})

# ---------- Ensure 'id' exists in both ----------
if "id" not in df_true.columns or "id" not in df_pred.columns:
    raise KeyError("Both CSVs must contain an 'id' column for matching (patient id).")

# ---------- Merge on id ----------
df_merged = pd.merge(df_pred, df_true, on="id", how="inner", suffixes=("_pred", "_true"))
if df_merged.empty:
    raise ValueError("Merge resulted in 0 rows. Check that 'id' values match between files.")

print(f"Merged {len(df_merged)} rows on 'id'.")

# ---------- Normalize label strings ----------
df_merged["pred_label_norm"] = df_merged["pred_label"].astype(str).str.strip().str.lower()
df_merged["actual_label_norm"] = df_merged["actual_label"].astype(str).str.strip().str.lower()
df_merged["label_match"] = df_merged["pred_label_norm"] == df_merged["actual_label_norm"]

# ---------- Numeric MMSE ----------
df_merged["pred_mmse"] = pd.to_numeric(df_merged["pred_mmse"], errors="coerce")
df_merged["actual_mmse"] = pd.to_numeric(df_merged["actual_mmse"], errors="coerce")
df_merged["mmse_diff"] = (df_merged["pred_mmse"] - df_merged["actual_mmse"]).abs()

# ---------- Metrics ----------
label_accuracy = df_merged["label_match"].mean() * 100
mae = df_merged["mmse_diff"].mean()
corr = df_merged["pred_mmse"].corr(df_merged["actual_mmse"])

# Optional: per-class precision/recall/f1
labels = sorted(set(df_merged["actual_label_norm"].unique()) | set(df_merged["pred_label_norm"].unique()))
y_true = df_merged["actual_label_norm"]
y_pred = df_merged["pred_label_norm"]
precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, labels=labels, zero_division=0)

prf_df = pd.DataFrame({
    "label": labels,
    "precision": precision,
    "recall": recall,
    "f1": f1,
    "support": support
})

# ---------- Print summary ----------
print("\n📊 Evaluation Results")
print(f"Samples evaluated: {len(df_merged)}")
print(f"Label Accuracy: {label_accuracy:.2f}% ({int(df_merged['label_match'].sum())}/{len(df_merged)})")
print(f"MMSE Mean Absolute Error (MAE): {mae:.3f}")
print(f"MMSE Pearson correlation: {corr:.3f}\n")

print("Per-class precision/recall/f1:")
print(prf_df.to_string(index=False))

# ---------- Save merged CSV ----------
out_csv = r"D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass_vs_actual_evaluation.csv"
df_merged.to_csv(out_csv, index=False)
print(f"\nSaved merged evaluation CSV: {out_csv}")

# ---------- Confusion matrix ----------
cm = confusion_matrix(df_merged["actual_label_norm"], df_merged["pred_label_norm"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)

plt.figure(figsize=(8,6))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix (normalized labels)")
confusion_path = r"D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass_confusion_matrix_labels.png"
plt.savefig(confusion_path, bbox_inches="tight")
plt.close()
print(f"Saved confusion matrix to: {confusion_path}")

# ---------- Scatter plot: actual vs predicted MMSE ----------
plt.figure(figsize=(6,6))
plt.scatter(df_merged["actual_mmse"], df_merged["pred_mmse"], alpha=0.7)
xmin = min(df_merged["actual_mmse"].min(), df_merged["pred_mmse"].min())
xmax = max(df_merged["actual_mmse"].max(), df_merged["pred_mmse"].max())
plt.plot([xmin, xmax], [xmin, xmax], linestyle="--", color="gray")
plt.xlabel("Actual MMSE")
plt.ylabel("Predicted MMSE")
plt.title("Predicted vs Actual MMSE")
plt.grid(True)
scatter_path = r"D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass_mmse_scatter.png"
plt.savefig(scatter_path, bbox_inches="tight")
plt.close()
print(f"Saved MMSE scatter plot to: {scatter_path}")

# ---------- Save per-class PRF ----------
prf_out = r"D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass_per_class_prf.csv"
prf_df.to_csv(prf_out, index=False)
print(f"Saved per-class precision/recall/f1 to: {prf_out}")

# ---------- Small summary file ----------
summary = {
    "n_samples": [len(df_merged)],
    "label_accuracy_pct": [label_accuracy],
    "mmse_mae": [mae],
    "mmse_corr": [corr]
}
pd.DataFrame(summary).to_csv(r"D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass_eval_summary.csv", index=False)
print("Saved eval summary to: D:\Dementia-Thesis - Don't access without permission\Thesis\qwen3-4B_fewshot_guidedprompt_multiexample_binaryclass_eval_summary.csv")